In [ ]:
import pandas as pd
from collections import Counter
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
import sys

In [ ]:
# MAX LEN IS PICKED BY LOOKING AT THE MEAN TOKEN LENGTH IN sentence_length.ipynb
MAX_LEN = 25

df = pd.read_csv("train_binary.csv")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["input_text"], df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [3]:
# will implement spaCy later
def tokenizer(text):
    return text.lower().split()

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(tokenizer(text))

vocab = {"<PAD>": 0, "<UNK>": 1}
for i, (word, _) in enumerate(counter.most_common(10000), start=2):
    vocab[word] = i

In [ ]:
def encode(text, vocab):
    tokens = tokenizer(text)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

In [17]:
def pad_sequence(seq, max_len=MAX_LEN):
    seq = seq[:max_len]
    return seq + [0] * (max_len - len(seq))

In [30]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = torch.tensor(labels.values, dtype=torch.float32)
        self.data = [
            pad_sequence(encode(text, vocab), MAX_LEN)
            for text in texts
        ]
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.data[idx], dtype=torch.long),
            self.labels[idx]
        )
    
train_dataset = TextDataset(train_texts, train_labels, vocab)
val_dataset   = TextDataset(val_texts,   val_labels,   vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# EMBEDING CLASSIFIER

In [ ]:
print(sys.executable)
print(torch.__file__)
print(torch.__version__)
print(hasattr(torch, "_utils"))

/home/hugs/py3.13venv/bin/python
/home/hugs/py3.13venv/lib/python3.12/site-packages/torch/__init__.py
2.10.0+cu128
True


In [ ]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):

        # x: [B, T]
        
        embeds = self.embedding(x) # x: [B,T,D]

        # FIRST UPDATE: FIX MASK
        mask = (x != 0).unsqueeze(-1)          # [B, T, 1]
        embeds = embeds * mask                 # zero-out PAD embeddings
        lengths = mask.sum(dim=1).clamp(min=1) # [B, 1]

        pooled = embeds.sum(dim=1) / lengths   # [B, D]

        logits = self.classifier(pooled)

        return logits.squeeze(1)

In [ ]:
model = EmbeddingClassifier(len(vocab))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_dataset = TextDataset(
    texts=df["input_text"],
    labels=df["label"],
    vocab=vocab
)

23993
(tensor([  15, 6193,    2, 2590,    2,   78,    4,    2, 2702,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0]), tensor(0.))


In [26]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TextDataset(df["input_text"], df["label"], vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = EmbeddingClassifier(len(vocab)).to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5 | Loss: 0.6049
Epoch 2/5 | Loss: 0.5471
Epoch 3/5 | Loss: 0.5133
Epoch 4/5 | Loss: 0.4868
Epoch 5/5 | Loss: 0.4621


In [33]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)  # logits
        preds = (torch.sigmoid(outputs) > 0.5)
        labels_bool = (labels > 0.5)

        correct += (preds == labels_bool).sum().item()
        total += labels.size(0)

print("Val Accuracy:", correct / total)

Val Accuracy: 0.7953740362575537


In [ ]:
#TODO ADD IN HANDCRAFTED FEATURES FOR BETTER MODEL ACCURACY

#ADD CHECKPOINTING SO WE CAN REUSE A TRAINED MODEL ON THE ACTUAL TEST SET